### Importing the weights of the YOLO model trained to find the bounding boxes

In [1]:
# Install gdown if it isn't already there
!pip install gdown -U

import gdown

# Paste YOUR specific File ID here
file_id = '15cKoNkawiQcGTX0XKualw-dIKCdtf2SR'

# Download the file and name it 'best.pt'
print("Downloading YOLO weights...")
gdown.download(id=file_id, output='best.pt', quiet=False)
print("Download complete!")

Downloading...
From: https://drive.google.com/uc?id=15cKoNkawiQcGTX0XKualw-dIKCdtf2SR
To: /content/best.pt
100%|██████████| 6.25M/6.25M [00:00<00:00, 236MB/s]

Download complete!


### Installation of paddle ocr

In [2]:
# 1. Wipe out any broken or conflicting packages
!pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu paddlex

# 2. Install YOLO, Gradio, and the STRICTLY STABLE version of PaddleOCR
!pip install ultralytics gradio paddleocr==2.9.1

# 3. Force-install the stable Paddle engine to prevent driver crashes
!python -m pip install paddlepaddle-gpu==2.6.1

Found existing installation: paddleocr 2.9.1
Uninstalling paddleocr-2.9.1:
  Successfully uninstalled paddleocr-2.9.1
  Using cached paddleocr-2.9.1-py3-none-any.whl.metadata (8.5 kB)
Using cached paddleocr-2.9.1-py3-none-any.whl (544 kB)
  Using cached paddlepaddle_gpu-2.6.1-cp312-cp312-manylinux1_x86_64.whl.metadata (8.6 kB)
  Using cached astor-0.8.1-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but yo

### Gradio for UI installation

In [3]:
!pip install gradio

### OCR output correction to Indian Liscence Plates format

In [4]:
import cv2
import os
import re
import pandas as pd
import logging
import gradio as gr
from ultralytics import YOLO
from paddleocr import PaddleOCR
from google.colab import drive


# 2. Strict Format Mask Function (Dynamic for any length)
TO_NUMBER = {
    "O": "0",
    "D": "0",
    "Q": "0",
    "I": "1",
    "L": "1",
    "T": "1",
    "Z": "2",
    "A": "4",
    "S": "5",
    "G": "6",
    "B": "8",
}
TO_LETTER = {"0": "O", "1": "I", "2": "Z", "4": "A", "5": "S", "6": "G", "8": "B"}

OCR_VISUAL_GROUPS = [
    {"O", "0", "D", "Q", "U", "C", "G"},
    {"1", "I", "l", "L", "T", "J", "7"},
    {"8", "B", "S", "3"},
    {"5", "S"},
    {"2", "Z", "7"},
    {"A", "4", "H", "R"},
    {"E", "F", "P", "B"},
    {"M", "W", "N", "V"},
    {"K", "X", "Y"},
    {"6", "G", "b", "C"},
    {"9", "P", "g", "q"}
]

def _visual_substitution_cost(char_a, char_b):
    if char_a == char_b:
        return 0
    for group in OCR_VISUAL_GROUPS:
        if char_a in group and char_b in group:
            return 0.5
    return 1

INDIAN_RTO_MAP = {
    "AN": ["01", "02", "03"],
    "AP": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "07",
        "09",
        "10",
        "11",
        "12",
        "13",
        "15",
        "16",
        "18",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "35",
        "36",
        "37",
        "39",
        "40",
    ],
    "AR": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "19",
        "20",
        "22",
    ],
    "AS": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
    ],
    "BR": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "19",
        "21",
        "22",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "37",
        "38",
        "39",
        "43",
        "44",
        "45",
        "46",
        "50",
        "51",
        "52",
        "53",
        "55",
        "56",
        "57",
    ],
    "CH": ["01", "02", "03", "04"],
    "CG": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
    ],
    "DD": ["01", "02", "03"],
    "DL": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13"],
    "DN": ["01", "02"],
    "GA": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
    "GJ": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
    ],
    "HR": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "88",
        "89",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
        "98",
        "99",
    ],
    "HP": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "88",
        "89",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
        "98",
        "99",
    ],
    "JH": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
    ],
    "JK": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "08",
        "09",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
    ],
    "KA": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
    ],
    "KL": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
    ],
    "LA": ["01", "02"],
    "LD": ["01"],
    "MH": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
    ],
    "ML": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10"],
    "MN": ["01", "02", "03", "04", "05", "06", "07", "08"],
    "MP": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
    ],
    "MZ": ["01", "02", "03", "04", "05", "06", "07", "08"],
    "NL": ["01", "02", "03", "04", "05", "06", "07", "08"],
    "OD": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
    ],
    "OR": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "14", "15", "16", "17"],
    "PB": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "88",
        "89",
        "90",
        "91",
    ],
    "PY": ["01", "02", "03", "04", "05"],
    "RJ": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
    ],
    "SK": ["01", "02", "03", "04", "05", "06", "07", "08"],
    "TG": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
    ],
    "TN": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "28",
        "30",
        "31",
        "32",
        "33",
        "34",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "81",
        "82",
        "84",
        "85",
        "86",
        "87",
        "88",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
        "99",
    ],
    "TR": ["01", "02", "03", "04", "05", "06", "07", "08"],
    "TS": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
    ],
    "UA": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
    "UK": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
    ],
    "UP": [
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
    ],
    "WB": [
        "01",
        "02",
        "03",
        "04",
        "05",
        "06",
        "07",
        "08",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "88",
        "89",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
        "98",
        "99",
    ],
}

INDIAN_RTO_MAP = {state: set(codes) for state, codes in INDIAN_RTO_MAP.items()}
STATE_CODES = sorted(code for code in INDIAN_RTO_MAP.keys())


def _normalize_text(raw_text):
    return re.sub(r"[^A-Z0-9]", "", raw_text.upper())

def _format_standard_plate(clean_text):
    fixed_text = ""
    length = len(clean_text)

    for i, char in enumerate(clean_text):
        if i < 2:
            fixed_text += TO_LETTER.get(char, char) if char.isdigit() else char
        elif i < 4:
            fixed_text += TO_NUMBER.get(char, char) if char.isalpha() else char
        elif length >= 8 and i >= length - 4:
            fixed_text += TO_NUMBER.get(char, char) if char.isalpha() else char
        else:
            fixed_text += TO_LETTER.get(char, char) if char.isdigit() else char
    return fixed_text

def _levenshtein_distance(a, b):
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            cost = _visual_substitution_cost(ca, cb)
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost))
        prev = curr
    return prev[-1]


def _closest_code(target, candidates, max_distance=1):
    best = None
    best_distance = max_distance + 1
    best_count = 0

    for code in candidates:
        distance = _levenshtein_distance(target, code)
        if distance < best_distance:
            best = code
            best_distance = distance
            best_count = 1
        elif distance == best_distance:
            best_count += 1

    if best_distance <= max_distance and best_count == 1:
        return best
    return None


def _apply_semantic_correction(plate_text):
    if len(plate_text) < 4:
        return plate_text, False

    state_code = plate_text[:2]
    rto_code = plate_text[2:4]

    if state_code not in INDIAN_RTO_MAP:
        corrected_state = _closest_code(state_code, STATE_CODES)
        if corrected_state is None:
            return plate_text, True
        state_code = corrected_state

    rto_codes = INDIAN_RTO_MAP.get(state_code, set())
    if rto_code not in rto_codes:
        corrected_rto = _closest_code(rto_code, rto_codes)
        if corrected_rto is None:
            return plate_text, True
        rto_code = corrected_rto

    corrected_text = plate_text
    if state_code != plate_text[:2] or rto_code != plate_text[2:4]:
        corrected_text = state_code + rto_code + plate_text[4:]

    return corrected_text, False


def _normalize_and_validate_plate(raw_text):
    clean_text = _normalize_text(raw_text)
    if not clean_text:
        return "", False

    fixed_text = _format_standard_plate(clean_text)
    return _apply_semantic_correction(fixed_text)


def enforce_strict_plate_format(raw_text):
    fixed_text, _ = _normalize_and_validate_plate(raw_text)
    return fixed_text


# 3. New Evaluation Logic (> 12 chars = Flag)
def evaluate_plate(plate_text):
    clean_text = plate_text.replace(" ", "").replace("|", "")
    if clean_text == "UNREADABLE" or clean_text == "NOPLATEDETECTED":
        return "Failed to read."

    for candidate in [part.strip() for part in plate_text.split("|") if part.strip()]:
        _, flagged = _normalize_and_validate_plate(candidate)
        if flagged:
            return "FLAGGED: Invalid State/RTO semantics."

    if len(clean_text) > 12:
        return "FLAGGED FOR MANUAL CHECK: Detected more than 12 characters."

    return "Processed (Note: OCR predictions may still not be 100% perfect)."


# 4. Load Models
print("Loading YOLO & PaddleOCR...")
# Load the model directly from the downloaded file
model_path = "best.pt"
yolo_model = YOLO(model_path)
ocr = PaddleOCR(use_angle_cls=False, lang='en', use_gpu=False)
print("Models loaded successfully!")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(


Loading YOLO & PaddleOCR...
download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:22<00:00, 175.42it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:02<00:00, 3715.88it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:20<00:00, 102.61it/s]

[2026/05/06 10:49:07] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec_c

Models loaded successfully!


In [ ]:
# 5. Core Processing Logic
def process_single_image(img_rgb):
    results = yolo_model(img_rgb, verbose=False)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    plate_found = False

    extracted_plates = [] # Safely store all plates found in the image

    for r in results:
        for box in r.boxes:
            plate_found = True
            x1, y1, x2, y2 = box.xyxy[0].int().tolist()
            cropped_plate = img_rgb[y1:y2, x1:x2]

             # --- ADVANCED PREPROCESSING FOR OCR ---
            # 1. Convert to Grayscale
            gray_plate = cv2.cvtColor(cropped_plate, cv2.COLOR_RGB2GRAY)

            # 2. Resize (Upscale by 2x using Cubic interpolation for smoother edges)
            resized_plate = cv2.resize(gray_plate, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

            # 3. Apply CLAHE (Locally enhances contrast to fight shadows and glare)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            contrast_plate = clahe.apply(resized_plate)

            # 4. Slight Gaussian Blur (Removes tiny speckles of dirt or noise)
            # A 3x3 blur is small enough to keep letters sharp but removes "salt and pepper" noise
            final_processed_plate = cv2.GaussianBlur(contrast_plate, (3, 3), 0)

            ### OCR

            paddle_results = ocr.ocr(final_processed_plate, cls=False)


            raw_text = ""
            if paddle_results and paddle_results[0] is not None:
                for line in paddle_results[0]:
                    raw_text += line[1][0]

                final_plate_text = enforce_strict_plate_format(raw_text)
                extracted_plates.append(final_plate_text) # Save the corrected version

                # Draw on image
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_bgr, final_plate_text, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            else:
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_bgr, "UNREADABLE", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Determine what to show in the UI text box
    if len(extracted_plates) > 0:
        final_output_text = " | ".join(extracted_plates)
    elif plate_found:
        final_output_text = "UNREADABLE"
    else:
        final_output_text = "NO PLATE DETECTED"
        cv2.putText(img_bgr, "NO PLATE DETECTED", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    status = evaluate_plate(final_output_text)
    annotated_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return annotated_rgb, final_output_text, status

def process_batch(files):
    results_data = []

    for file_obj in files:
        file_path = file_obj.name
        file_name = os.path.basename(file_path)

        img = cv2.imread(file_path)
        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        _, final_plate_text, status = process_single_image(img_rgb)

        results_data.append({
            "Image Name": file_name,
            "Extracted Plate": final_plate_text,
            "Evaluation Status": status
        })

    df = pd.DataFrame(results_data)
    csv_path = "/content/batch_results.csv"
    df.to_csv(csv_path, index=False)

    return df, csv_path
# 7. Build the Gradio UI
gr.close_all()
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🚘 Automatic License Plate Recognition System")
    gr.Markdown("Upload images to detect license plates. If the extracted text exceeds 12 characters, it will be flagged for manual review.")

    with gr.Tabs():
        # TAB 1: Single Image Mode
        with gr.TabItem("Single Image Analyzer"):
            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(label="Upload Car Image")
                    analyze_btn = gr.Button("Analyze Plate", variant="primary")
                with gr.Column():
                    image_output = gr.Image(label="Annotated Result")
                    text_output = gr.Textbox(label="Extracted Plate Number", text_align="center")
                    status_output = gr.Textbox(label="Evaluation Status")

            analyze_btn.click(
                fn=process_single_image,
                inputs=image_input,
                outputs=[image_output, text_output, status_output]
            )

        # TAB 2: Batch / Folder Mode
        with gr.TabItem("Batch/Folder Analyzer"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("Select multiple files from your computer to process them all at once.")
                    file_input = gr.File(file_count="multiple", label="Upload Multiple Images")
                    batch_btn = gr.Button("Process Batch", variant="primary")
                with gr.Column():
                    dataframe_output = gr.Dataframe(label="Batch Results Preview")
                    csv_output = gr.File(label="Download CSV Results")

            batch_btn.click(
                fn=process_batch,
                inputs=file_input,
                outputs=[dataframe_output, csv_output]
            )

# Launch the app right inside Colab!
app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a379dce6e2f8ae6f7d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

[2026/05/06 10:51:51] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.11418533325195312
[2026/05/06 10:51:51] ppocr DEBUG: rec_res num  : 1, elapsed : 0.16208600997924805
[2026/05/06 10:51:52] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.05227184295654297
[2026/05/06 10:51:52] ppocr DEBUG: rec_res num  : 1, elapsed : 0.0596919059753418
[2026/05/06 10:51:52] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.014830350875854492
[2026/05/06 10:51:52] ppocr DEBUG: rec_res num  : 1, elapsed : 0.05997467041015625
[2026/05/06 10:51:52] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.011612892150878906
[2026/05/06 10:51:52] ppocr DEBUG: rec_res num  : 1, elapsed : 0.05946469306945801
[2026/05/06 10:51:53] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.020402908325195312
[2026/05/06 10:51:53] ppocr DEBUG: rec_res num  : 1, elapsed : 0.09723639488220215
[2026/05/06 10:51:53] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.09124159812927246
[2026/05/06 10:51:53] ppocr DEBUG: rec_res num  : 1, elapsed : 0.08979368209838867
[2